# LoRA Fine-Tuning Assignment

## Objective

Apply what you learned in the SFT + LoRA tutorial to fine-tune **TinyLlama-1.1B** on the **databricks-dolly-15k** dataset using LoRA.

## What is provided

- All installation, import, and utility cells are provided.
- Each task cell has a comment block describing exactly what you need to write.

## What you need to do

Fill in every cell marked with `# YOUR CODE HERE`. There are **8 tasks** in total.

## Prerequisites

- You have completed the SFT + LoRA tutorial notebook.
- Runtime: **GPU** (T4 is sufficient).

---
## Part 0 — Setup (provided)

In [1]:
!pip install -U transformers trl peft datasets accelerate -q

In [2]:
#!pip uninstall -y torchao

In [11]:
import torch
import gc
import os
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTConfig, SFTTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


### Utility functions (provided)

In [4]:
def clear_memory():
    """Free GPU memory between experiments."""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


def print_trainable_params(model):
    """Display trainable vs total parameter counts."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:>12,}  ({100 * trainable / total:.4f}%)")
    print(f"Total parameters:     {total:>12,}")


def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    """Generate a response from a model given a prompt string."""
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

---
## Part 1 — Configuration

### Task 1: Define all hyperparameters

Set the following configuration variables:

| Variable | Value |
|---|---|
| `MODEL_NAME` | `"TinyLlama/TinyLlama-1.1B-Chat-v1.0"` |
| `DATASET_NAME` | `"databricks/databricks-dolly-15k"` |
| `NUM_TRAIN_SAMPLES` | `1000` |
| `MAX_SEQ_LENGTH` | `512` |
| `LORA_R` | `8` |
| `LORA_ALPHA` | `16` |
| `LORA_DROPOUT` | `0.05` |
| `LORA_TARGET_MODULES` | `["q_proj", "v_proj"]` (attention only) |
| `LEARNING_RATE` | `2e-4` |
| `NUM_EPOCHS` | `1` |
| `BATCH_SIZE` | `4` |
| `GRAD_ACCUM_STEPS` | `4` |
| `OUTPUT_DIR` | `"./sft-lora-tinyllama"` |

In [5]:
#from transformers.models.canine.tokenization_canine import MASK
# Part1 Configuration
# Model and Dataset configuration

# Pretrained casual language model used as the base model
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

#Instruction-following dataset used for Supervised Fine Tuning
DATASET_NAME = "databricks/databricks-dolly-15k"

#Number of Training samples selected from Dolly-15K
NUM_TRAIN_SAMPLES = 1000

#Maximum no of tokens allowed for each training sequence
MAX_SEQ_LENGTH = 512

# LoRA configuration

# Smaller rank for fewer trainable parameter and lower memory use
LORA_R = 8

# Effective LORA scaling is approximately LORA_ALPHA /LORA_R
LORA_ALPHA = 16

# Dropout applied to LORA adapters for regularization
LORA_DROPOUT = 0.05

# Apply LORA for query and value projection layers
LORA_TARGET_MODULES = ["q_proj", "v_proj"]


#Training Configurations

#Learning Rate
LEARNING_RATE = 2e-4

#Number of complete passes through the training dataset
NUM_EPOCHS =1

#Num of samples processed per device in one forward pass
BATCH_SIZE = 4

# Accumulate gradients across multiple mini-batches before performing one optimizer update.
GRAD_ACCUM_STEPS = 4

# Directory used for checkpoints and final adapter artifacts.
OUTPUT_DIR = "./sft-lora-tinyllama"


---
## Part 2 — Load Model and Tokenizer

### Task 2: Load the tokenizer and base model

1. Load the tokenizer from `MODEL_NAME`.
2. If the tokenizer has no pad token, set it to the eos token.
3. Load the model in `float16` with `device_map="auto"`.
4. Print the total parameter count.

In [6]:
# Load the tokenizer associated with the TinyLLama checkpoint

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#TinyLlama may not define a separate padding token. For causal language models, using the EOS token as padding

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the pre-trained TinyLlama model in FP16.
# FP16 reduces GPU memory usage compared with FP32.
# device_map="auto" places the model on the available GPU automatically.

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16, #means model parameters are loaded primarily in 16-bit floating point instead of FP32.
    device_map="auto",
)


#Calculate the total number of parameters in the base model
total_parameters = sum(parameter.numel() for parameter in model.parameters())
print(f"Total parameters: {total_parameters:,}")
print(f"Model device: {next(model.parameters()).device}")
print(f"Model dtype : {next(model.parameters()).dtype}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Total parameters: 1,100,048,384
Model device: cuda:0
Model dtype : torch.float16


---
## Part 3 — Test Base Model

### Task 3: Generate responses from the base model

Use the provided `generate_response` function with these test prompts:
1. `"Explain what a neural network is in 2-3 sentences."`
2. `"What are three tips for better sleep?"`

Store the responses in a list called `base_responses` for later comparison.

In [7]:
# ============================================================
# PART 3: TEST BASE MODEL
# ============================================================

# Fixed evaluation prompts used to measure the behavious of the base model before LORA fine tuning
test_prompts =["Explain what a neural network is in 2-3 sentences.",
               "What are three tips for better sleep?"]

# Store the base responses to compare the LoRA fine tuned model later
base_responses = []


for prompt in test_prompts:

    response = generate_response(model = model,
                                 tokenizer = tokenizer,
                                 prompt = prompt)

    base_responses.append(response)

    print ("=" *50)
    print (f"Test Prompt:'\n{prompt}")
    print ("*" *50)
    print (f"Test Prompt Response:\n{response}")
    print ("*" *50)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test Prompt:'
Explain what a neural network is in 2-3 sentences.
**************************************************
Test Prompt Response:
A neural network is a computational model that mimics the structure and functions of the human brain. It consists of a series of interconnected computational units (neurons) that can process information and make decisions based on previous experiences and data. Neural networks can be used in a variety of applications, such as image recognition, speech recognition, and machine learning. They are designed to learn from data and improve their performance over time through iterative learning processes.
**************************************************
Test Prompt:'
What are three tips for better sleep?
**************************************************
Test Prompt Response:
1. Create a sleep-friendly bedtime routine: This can include avoiding electronic devices an hour before bedtime and aiming for a consistent bedtime routine.

2. Establish a sleep sch

---
## Part 4 — Prepare the Dataset

### Task 4: Load and format the Dolly dataset

The Dolly dataset has columns: `instruction`, `context`, `response`, `category`.

1. Load the dataset (`split="train"`).
2. Write a function `format_dolly_to_chat(example)` that converts each example to:
   ```python
   {"messages": [
       {"role": "user", "content": <instruction + context if present>},
       {"role": "assistant", "content": <response>}
   ]}
   ```
   If `context` is non-empty, concatenate it to the instruction with a newline separator.
3. Shuffle with `seed=42`, select `NUM_TRAIN_SAMPLES` examples, and apply the formatting function.
4. Print the size of the formatted dataset and one example.



```
Example formatted entry:
{'messages': [{'content': 'Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?', 'role': 'user'}, {'content': 'Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree', 'role': 'assistant'}]}
```



In [8]:
# ============================================================
# PART 4: PREPARE THE DOLLY DATASET
# ============================================================
# Load the dataset

raw_dataset = load_dataset(DATASET_NAME, split="train")

# ------------------------------------------------------------
# Convert Dolly records into chat format
# ------------------------------------------------------------

def format_dolly_to_chat(example):

  """
    Convert a Dolly instruction-response record into the chat structure expected for supervised fine-tuning.

    If context is available, it is appended to the instruction using a newline separator.

    Parameters
    ----------
    example : dict
        A single Dolly dataset record containing: instruction, context, response, and category.

    Returns
    -------
    dict
        Dictionary containing a 'messages' field with user and assistant messages.
    """

  instruction = example["instruction"].strip()
  context = example["context"].strip()
  response = example["response"].strip()





  if len(context) > 0:
    user_content = f"{instruction}\n{context}"
  else:
    user_content = instruction

  return {"messages": [
      {"role": "user", "content": user_content},
      {"role": "assistant", "content": response}
  ]}


  # ------------------------------------------------------------
  # Step 3: Shuffle, sample, and format the dataset
  # ------------------------------------------------------------

sample_dataset = (raw_dataset.shuffle(seed=42).select(range(NUM_TRAIN_SAMPLES)))

formatted_dataset = sample_dataset.map(format_dolly_to_chat, remove_columns = sample_dataset.column_names)

# ------------------------------------------------------------

assert len(formatted_dataset) == NUM_TRAIN_SAMPLES, \
    "Formatted dataset size does not match NUM_TRAIN_SAMPLES."

assert "messages" in formatted_dataset.column_names, \
    "Expected 'messages' column was not created."

assert len(formatted_dataset[0]["messages"]) == 2, \
    "Each training example must contain one user and one assistant message."

print(f"Formatted dataset size: {len(formatted_dataset)}")
print("\nExample formatted entry:")
print(formatted_dataset[0])
print("\nDataset validation passed.")

Formatted dataset size: 1000

Example formatted entry:
{'messages': [{'content': 'Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?', 'role': 'user'}, {'content': 'Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree', 'role': 'assistant'}]}

Dataset validation passed.


---
## Part 5 — Configure LoRA

### Task 5: Create the LoRA config and inspect trainable parameters

1. Create a `LoraConfig` using the hyperparameters from Task 1. Set `bias="none"` and `task_type="CAUSAL_LM"`.
2. Apply it to the model with `get_peft_model`.
3. Use `print_trainable_params` to display the trainable vs total parameters.
4. Print the LoRA-wrapped structure of one attention layer (e.g., `q_proj` of layer 0).

In [9]:
#LORA Configuration

# ------------------------------------------------------------
# Step 1: Define the LoRA configuration
# ------------------------------------------------------------


lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

# Insert the LORA Adapters into the Base Model

peft_model = get_peft_model(model, lora_config)

# ------------------------------------------------------------
# Step 3: Display the trainable parameters
# ------------------------------------------------------------

print_trainable_params(peft_model)


# ------------------------------------------------------------
# Step 4: Inspect one LoRA-wrapped attention projection layer
# ------------------------------------------------------------

print("\nLoRA-wrapped q_proj from layer 0:")

print(peft_model.base_model.model.model.layers[0].self_attn.q_proj)


Trainable parameters:    1,126,400  (0.1023%)
Total parameters:     1,101,174,784

LoRA-wrapped q_proj from layer 0:
lora.Linear(
  (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.05, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=2048, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=2048, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)


---
## Part 6 — Train

### Task 6: Set up SFTTrainer and run training

1. Delete the PEFT-wrapped model and call `clear_memory()`. Reload the base model fresh (SFTTrainer applies LoRA itself via `peft_config`).
2. Create an `SFTConfig` with the hyperparameters from Task 1. Include:
   - Cosine LR scheduler, warmup ratio 0.03, weight decay 0.01
   - `gradient_checkpointing=True`
   - bf16 if supported, else fp16
   - `max_length=MAX_SEQ_LENGTH`
   - Logging every 10 steps
3. Create an `SFTTrainer` with the model, training args, dataset, tokenizer, and `peft_config`.
4. Call `trainer.train()` and print the final loss.

In [28]:
# Part 6 - Train the LoRA Adapter
# ------------------------------------------------------------
# Step 1: Remove the inspection PEFT model and clear GPU memory
# ------------------------------------------------------------
if "peft_model" in globals():
    del peft_model

if "model" in globals():
    del model

clear_memory()

#use BF16 when the GPU supports it; otherwise fall back to FP16

bf16_supported = torch.cuda.is_bf16_supported()
fp16_supported = not bf16_supported

training_dtype = (
    torch.bfloat16
    if bf16_supported
    else torch.float16
)

print(f"BF16 supported: {bf16_supported}")
print(f"Training precision: {'BF16' if bf16_supported else 'FP16'}")

# ------------------------------------------------------------
# Step 3: Reload a fresh TinyLlama base model
# ------------------------------------------------------------

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Gradient checkpoint is used during Training, so disable KV-Cache because cache is incompatible with this set-up

model.config.use_cache = False


# ------------------------------------------------------------
# Step 4: Configure Supervised Fine Tuning
# ------------------------------------------------------------

sft_config = SFTConfig(
    output_dir = OUTPUT_DIR,
    learning_rate = LEARNING_RATE,
    num_train_epochs=  NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    #Learning Rate Scheduling and regularization

    lr_scheduler_type ="cosine",
    warmup_steps = 0.03,
    weight_decay=0.01,

    #Memory Optimization
    bf16 = bf16_supported,
    fp16 = fp16_supported,

    #Maximum tokenized sequence length
    max_length = MAX_SEQ_LENGTH,


    # Training progress reporting

    logging_steps = 10,
    logging_strategy ="steps",

    # Reproducibility
    seed = 42,

    # Avoid unnecessary external experiment trackers
    report_to="none"


)

# ------------------------------------------------------------
# Step 5: Create the SFTTrainer
# ------------------------------------------------------------

trainer = SFTTrainer(
    model = model,
    train_dataset = formatted_dataset,
    # SFTTrainer injects the LoRA adapters into the clean base model using this configuration.
    peft_config = lora_config,
    # Current TRL uses processing_class instead of the older tokenizer argument.
    processing_class = tokenizer,
    args = sft_config,)



# ------------------------------------------------------------
# Step 6: Verify trainable parameters before training
# ------------------------------------------------------------

print_trainable_params(trainer.model)

# ------------------------------------------------------------
# Step 7: Run training
# ------------------------------------------------------------

train_result = trainer.train()


# ------------------------------------------------------------
# Step 8: Report the final training loss
# ------------------------------------------------------------

final_loss = train_result.training_loss

print("*" *50)
print("Training complete.")
print("*" *50)
print(f"Final training loss: {final_loss}")
print("*" *50)

GPU allocated: 0.01 GB
GPU reserved:  2.05 GB
BF16 supported: True
Training precision: BF16


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4117 > 2048). Running this sequence through the model will result in indexing errors


Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Trainable parameters:    1,126,400  (0.1023%)
Total parameters:     1,101,174,784


Step,Training Loss
10,2.113773
20,1.881631
30,1.760928
40,1.736523
50,1.782360
60,1.725796


**************************************************
Training complete.
**************************************************
Final training loss: 1.8327750175718278
**************************************************


---
## Part 7 — Save the Adapter

### Task 7: Save the LoRA adapter and print its size

1. Save the trained model to `f"{OUTPUT_DIR}/final-adapter"`.
2. Also save the tokenizer to the same path.
3. Compute and print the adapter size in MB.

In [29]:
# ============================================================
# PART 7: SAVE THE LoRA ADAPTER
# ============================================================

import os

# ------------------------------------------------------------
# Step 1: Define the final adapter directory
# ------------------------------------------------------------

final_adapter_path = f"{OUTPUT_DIR}/final-adapter"

# ------------------------------------------------------------
# Step 2: Save the trained model and tokenizer
# ------------------------------------------------------------

trainer.model.save_pretrained(final_adapter_path)

# ------------------------------------------------------------
# Step 3: Save the tokenizer
# ------------------------------------------------------------

tokenizer.save_pretrained(final_adapter_path)

# ------------------------------------------------------------
# Step 4: Compute and print the adapter size
# ------------------------------------------------------------

adapter_size = sum(
    os.path.getsize(os.path.join(final_adapter_path, file))
    for file in os.listdir(final_adapter_path)
    if os.path.isfile(os.path.join(final_adapter_path, file))
)

adapter_size_mb = adapter_size / (1024 **2)

print(f"Adapter size: {adapter_size_mb} MB")

# ------------------------------------------------------------
# Step 5: Display adapter information
# ------------------------------------------------------------

print(f"Adapter saved to : {final_adapter_path}")
print(f"Adapter size     : {adapter_size_mb:.2f} MB")


Adapter size: 7.765937805175781 MB
Adapter saved to : ./sft-lora-tinyllama/final-adapter
Adapter size     : 7.77 MB


---
## Part 8 — Evaluate

### Task 8: Compare base vs fine-tuned responses

1. Use `trainer.model` (already has LoRA weights) to generate responses for the same `test_prompts`.
2. Print a side-by-side comparison of base vs fine-tuned responses.

In [32]:
# ============================================================
# PART 8: EVALUATE BASE VS FINE-TUNED MODEL
# ============================================================

# ------------------------------------------------------------
# Step 1: Prepare the fine-tuned model for inference
# ------------------------------------------------------------

fine_tuned_LORA_model = trainer.model

# Switch from training mode to evaluation mode.

fine_tuned_LORA_model.eval()


# ------------------------------------------------------------
# Step 2: Generate responses using the same test prompts
# ------------------------------------------------------------

fine_tuned_responses = []

for prompt in test_prompts:

    response = generate_response(model = fine_tuned_LORA_model,
                                 tokenizer = tokenizer,
                                 prompt = prompt)

    fine_tuned_responses.append(response)



# ------------------------------------------------------------
# Step 3: Compare base and fine-tuned responses
# ------------------------------------------------------------

for index, prompt in enumerate(test_prompts):

    print ("=" *50)
    print (f"Test Prompt:'{index+1}':\n{prompt}")
    print ("*" *50)

    print (f"Base Model Response:\n{base_responses[index]}")
    print ("*" *50)

    print (f"Fine-Tuned Model Response:\n{fine_tuned_responses[index]}")
    print ("*" *50)

    print()





[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test Prompt:'1':
Explain what a neural network is in 2-3 sentences.
**************************************************
Base Model Response:
A neural network is a computational model that mimics the structure and functions of the human brain. It consists of a series of interconnected computational units (neurons) that can process information and make decisions based on previous experiences and data. Neural networks can be used in a variety of applications, such as image recognition, speech recognition, and machine learning. They are designed to learn from data and improve their performance over time through iterative learning processes.
**************************************************
Fine-Tuned Model Response:
A neural network is a model that can simulate the behavior of the brain. It is a collection of connected layers of neurons, each of which has an input, an output, and an activation function. The neurons can receive inputs from other neurons and pass on their own output to other

---
## Done

You have successfully fine-tuned TinyLlama with LoRA on the Dolly dataset. Review your base vs fine-tuned outputs and note the differences in instruction-following quality.